In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"
N8N_WEBHOOK = "http://localhost:5678/webhook-test/Clientes"

html = requests.get(URL_PAGINA, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

linhas = soup.select("#clientes tbody tr")
clientes = []

for linha in linhas:
    colunas = linha.find_all("td")
    clientes.append({
        "nome": colunas[0].get_text(strip=True),
        "email": colunas[1].get_text(strip=True),
        "saldo": colunas[2].get_text(strip=True),
        "perfil": colunas[3].get_text(strip=True)
    })

print("Clientes enviados:", len(clientes))

payload = {"clientes": clientes}

resp = requests.post(
    N8N_WEBHOOK,
    json=payload,
    headers={"Content-Type": "application/json"},
    timeout=60
)

print("Status:", resp.status_code)

try:
    dados = resp.json()

    if isinstance(dados, list):
        df = pd.json_normalize(dados)
    elif isinstance(dados, dict):
        df = pd.json_normalize(dados)
    else:
        df = pd.DataFrame(dados)

    print("\nDataFrame retornado pelo n8n:")
    print(df.to_string(index=False))

except Exception as e:
    print("Erro ao transformar resposta em DataFrame:", e)
    print("Resposta recebida:")
    print(resp.text[:3000])

Clientes encontrados: 10
Status: 200

DataFrame retornado pelo n8n:
           nome             produto rentabilidade
      Ana Silva            Poupança          6.2%
      Ana Silva CDB Liquidez Diária         12.5%
      Ana Silva       Tesouro Selic         13.0%
     Bruno Lima       CDB Prefixado         14.0%
     Bruno Lima       Tesouro IPCA+     IPCA+6.0%
    Carla Souza           Ações ETF      Variável
    Carla Souza      Fundo de Ações      Variável
    Carla Souza        Criptomoedas      Variável
Daniel Oliveira            Poupança          6.2%
Daniel Oliveira       Tesouro Selic         13.0%
Elena Rodrigues           Ações ETF      Variável
Elena Rodrigues      Fundo de Ações      Variável
Elena Rodrigues        Criptomoedas      Variável
  Felipe Santos       CDB Prefixado         14.0%
  Felipe Santos  Fundo Multimercado         16.0%
  Felipe Santos       Tesouro IPCA+     IPCA+6.0%
 Gabriela Costa            Poupança          6.2%
 Gabriela Costa CDB Liquidez Diá